# Preprocessing with scLucid

This notebook demonstrates the complete preprocessing workflow.

In [ ]:
import scanpy as sc
import scLucid
from scLucid.preprocess import WorkflowConfig, run_preprocessing

# Load QC-filtered data
adata = sc.read_h5ad("results/qc_filtered.h5ad")

## Configure Preprocessing Workflow

In [ ]:
preprocess_config = WorkflowConfig(
    # Normalization
    normalization={
        "target_sum": 1e4,
        "output_layer": "normalized"
    },
    # HVG selection
    hvg={
        "method": "seurat",
        "n_top_genes": 2000
    },
    # Scaling
    scaling={
        "scale_zero_center": True,
        "max_value": 10
    },
    # PCA
    graph={
        "n_pcs": 50
    },
    # Batch correction (if needed)
    integration={
        "method": None  # Set to "harmony" or "scanorama" if needed
    }
)

## Run Preprocessing Workflow

In [ ]:
adata_pp = run_preprocessing(
    adata,
    config=preprocess_config,
    save_dir="results/preprocess"
)

print(f"Normalized data: {adata_pp.layers['normalized'].shape}")
print(f"HVGs: {adata_pp.var['highly_variable'].sum()}")
print(f"PCs: {adata_pp.obsm['X_pca'].shape[1]}")
print(f"UMAP: {adata_pp.obsm['X_umap'].shape}")

## Visualize Results

In [ ]:
# PCA variance plot (saved by workflow)
from IPython.display import Image
Image('results/preprocess/pca_variance_ratio.png')

# UMAP plot
sc.pl.umap(adata_pp, color=['batch'], save='_preprocess.pdf')

## Save Results

In [ ]:
# Save preprocessed data
adata_pp.write('results/preprocessed.h5ad')

# Save config
preprocess_config.to_json_file('results/preprocess_config.json')